In [ ]:
# %% [markdown]
# # Encoding voor het MysteryDevice
# **Doel:** Een MNIST-afbeelding zo compact mogelijk encoderen zodat een Decision Tree
# tijdens *inference* zo min mogelijk RAM en rekenkracht nodig heeft.
#
# **MysteryDevice beperkingen:**
# - RAM: 256 KB
# - Opslag: 1 MB (programma + model samen)
# - Geen GPU, embedded Python

# %% [markdown]
# ## Setup – laad één MNIST sample

# %%
import numpy as np
from sklearn.datasets import fetch_openml

# Laad MNIST (klein gedeelte is genoeg voor demonstratie)
print("MNIST laden...")
mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto")
X, y = mnist.data, mnist.target

# Neem één sample en zet terug naar 28×28
sample_flat = X[0]                          # shape (784,), float64
sample = sample_flat.reshape(28, 28)        # 2D afbeelding
label  = y[0]

print(f"Label: {label}")
print(f"Origineel dtype  : {sample.dtype}")
print(f"Origineel bytes  : {sample.nbytes} bytes  ({sample.nbytes/1024:.2f} KB)")


# %% [markdown]
# ## Hulpfuncties

# %%
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

def show_comparison(images, titles, cmap="gray"):
    """Toon meerdere afbeeldingen naast elkaar."""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
    if n == 1:
        axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        # Zorg dat we altijd 2D kunnen plotten
        display = img.reshape(int(np.sqrt(img.size)), -1) if img.ndim == 1 else img
        ax.imshow(display, cmap=cmap, vmin=0, vmax=display.max() or 1)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

def report(name, encoded):
    """Print een nette samenvatting van een encoding."""
    print(f"\n{'─'*45}")
    print(f"  Methode  : {name}")
    print(f"  Shape    : {encoded.shape}")
    print(f"  dtype    : {encoded.dtype}")
    print(f"  Bytes    : {encoded.nbytes} bytes")
    print(f"  100 imgs : {encoded.nbytes * 100 / 1024:.2f} KB")
    print(f"{'─'*45}")


# %% [markdown]
# ## Methode 1 – Baseline (float64, flat)
# Geen encoding, gewoon flattenen. Dit is het startpunt.

# %%
def encode_baseline(img):
    return img.flatten().astype(np.float64)

enc_baseline = encode_baseline(sample)
report("Baseline float64 flat", enc_baseline)
show_comparison([sample], [f"Origineel (label={label})"])


# %% [markdown]
# ## Methode 2 – Float32 → al 2× kleiner

# %%
def encode_float32(img):
    return img.flatten().astype(np.float32)

enc_f32 = encode_float32(sample)
report("Float32 flat", enc_f32)


# %% [markdown]
# ## Methode 3 – uint8 quantization (8-bit)
# Pixelwaarden liggen al in [0, 255] → opslaan als 1 byte per pixel.

# %%
def encode_uint8(img):
    return img.astype(np.uint8).flatten()

enc_uint8 = encode_uint8(sample)
report("uint8 (8-bit)", enc_uint8)


# %% [markdown]
# ## Methode 4 – 4-bit quantization (16 niveaus)
# Twee pixels in één byte passen → 392 bytes voor 784 pixels.

# %%
def encode_4bit(img):
    pixels = (img / 255.0 * 15).astype(np.uint8).flatten()  # schaal naar 0-15
    # Pak twee pixels samen in één byte: hoge nibble + lage nibble
    paired = (pixels[0::2] << 4) | (pixels[1::2] & 0x0F)
    return paired.astype(np.uint8)

def decode_4bit(encoded, size=784):
    high = (encoded >> 4) & 0x0F
    low  = encoded & 0x0F
    pixels = np.empty(size, dtype=np.uint8)
    pixels[0::2] = high * 17   # 0→0, 15→255
    pixels[1::2] = low  * 17
    return pixels.reshape(28, 28)

enc_4bit = encode_4bit(sample)
report("4-bit quantization", enc_4bit)
show_comparison([sample, decode_4bit(enc_4bit)],
                ["Origineel", "4-bit gedecodeerd"])


# %% [markdown]
# ## Methode 5 – Binary thresholding (1 bit per pixel)
# Elke pixel is zwart of wit. 784 bits = **98 bytes**.

# %%
def encode_binary(img, threshold=127):
    binary = (img > threshold).flatten().astype(bool)
    return np.packbits(binary)   # pakt 8 booleans in 1 byte

def decode_binary(encoded, size=784):
    bits = np.unpackbits(encoded)[:size]
    return (bits * 255).reshape(28, 28).astype(np.uint8)

enc_binary = encode_binary(sample)
report("Binary thresholding (1-bit)", enc_binary)
show_comparison([sample, decode_binary(enc_binary)],
                ["Origineel", "Binair (zwart/wit)"])


# %% [markdown]
# ## Methode 6 – Downscaling 28×28 → 14×14
# Halveer resolutie door blokken van 4 pixels te middelen → 196 features.

# %%
def encode_downscale(img, factor=2):
    h, w = img.shape
    new_h, new_w = h // factor, w // factor
    reshaped = img[:new_h*factor, :new_w*factor]\
                  .reshape(new_h, factor, new_w, factor)
    return reshaped.mean(axis=(1, 3)).astype(np.uint8).flatten()

enc_down = encode_downscale(sample, factor=2)
report("Downscaling 14×14 uint8", enc_down)
show_comparison([sample, enc_down.reshape(14, 14)],
                ["Origineel 28×28", "Downscaled 14×14"])


# %% [markdown]
# ## Methode 7 – Ordinale binning (3 niveaus: donker / medium / licht)
# Pixels krijgen waarde 0, 1 of 2 → past in 2 bits, opgeslagen als uint8.

# %%
def encode_ordinal_bins(img):
    bins = np.digitize(img.flatten(), bins=[85, 170]) # 0-84=donker, 85-169=medium, 170-255=licht
    return bins.astype(np.uint8)

enc_bins = encode_ordinal_bins(sample)
report("Ordinale binning (3 niveaus)", enc_bins)
show_comparison([sample, enc_bins.reshape(28, 28)],
                ["Origineel", "3-waarden binning"])


# %% [markdown]
# ## Methode 8 – Feature Extraction (zone-inktverdeling)
# In plaats van alle pixels, bewaar slechts 9 features:
# hoeveel inkt zit er in elk van de 9 zones (3×3 grid)?

# %%
def encode_zone_features(img, grid=3):
    """Verdeel de afbeelding in grid×grid zones, geef gemiddelde inkt per zone."""
    h, w = img.shape
    zh, zw = h // grid, w // grid
    features = []
    for row in range(grid):
        for col in range(grid):
            zone = img[row*zh:(row+1)*zh, col*zw:(col+1)*zw]
            features.append(zone.mean())
    return np.array(features, dtype=np.float32)

enc_zones = encode_zone_features(sample, grid=3)
report("Zone-features (9 waarden)", enc_zones)
print("  Feature waarden:", np.round(enc_zones, 1))

# Visualiseer als 3×3 heatmap naast origineel
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(sample, cmap="gray"); axes[0].set_title("Origineel"); axes[0].axis("off")
axes[1].imshow(enc_zones.reshape(3, 3), cmap="hot")
axes[1].set_title("Zone-features (3×3)"); axes[1].axis("off")
plt.tight_layout(); plt.show()


# %% [markdown]
# ## Methode 9 – Eigen encoding: "Sparse + Binary + Downscale" combinatie
# **Idee:** Combineer downscaling (14×14) met binary thresholding.
# - Resultaat: 196 bits = **25 bytes** voor één afbeelding
# - Bewaard de globale vorm van het cijfer, weggooit onnodige grijstinten

# %%
def encode_custom(img, threshold=80):
    """
    Eigen encoding:
    1. Downscale 28×28 → 14×14  (middelen van 2×2 blokken)
    2. Binary threshold          (lage drempel: ook lichtgrijze pixels tellen mee)
    3. Packbits                  (196 bits → 25 bytes)
    """
    # Stap 1: downscale
    small = img.reshape(14, 2, 14, 2).mean(axis=(1, 3))
    # Stap 2: binary threshold
    binary = (small > threshold).flatten()
    # Stap 3: pack naar bytes
    return np.packbits(binary)

def decode_custom(encoded, size=196):
    bits = np.unpackbits(encoded)[:size]
    return (bits * 255).reshape(14, 14).astype(np.uint8)

enc_custom = encode_custom(sample)
report("Eigen encoding: 14×14 binary packed", enc_custom)
show_comparison([sample, decode_custom(enc_custom)],
                ["Origineel", "Eigen encoding gedecodeerd"])


# %% [markdown]
# ## Vergelijkingstabel

# %%
methods = [
    ("Baseline float64",        enc_baseline),
    ("Float32",                  enc_f32),
    ("uint8 (8-bit quant.)",     enc_uint8),
    ("4-bit quantization",       enc_4bit),
    ("Binary 1-bit packed",      enc_binary),
    ("Downscale 14×14 uint8",    enc_down),
    ("Ordinale binning",         enc_bins),
    ("Zone-features (9 vals)",   enc_zones),
    ("Eigen: 14×14 binary",      enc_custom),
]

baseline_bytes = enc_baseline.nbytes
RAM_LIMIT_KB   = 256 * 1024   # 256 KB in bytes

print(f"\n{'Methode':<28} {'Bytes':>6}  {'100 imgs':>10}  {'vs baseline':>12}  {'RAM-safe?':>10}")
print("─" * 75)
for name, enc in methods:
    b        = enc.nbytes
    per_100  = b * 100
    ratio    = baseline_bytes / b
    safe     = "✓" if per_100 < RAM_LIMIT_KB * 0.5 else "~"   # laat helft RAM over voor model
    print(f"{name:<28} {b:>6}  {per_100/1024:>8.2f} KB  {ratio:>10.1f}×  {safe:>10}")

print(f"\nRAM-limiet device : 256 KB = {256*1024} bytes")
print(f"Richtlijn 1 sample: < {256*1024 // 100} bytes zodat 100 samples < 128 KB past")


# %% [markdown]
# ## Antwoorden op de vragen
#
# | Vraag | Antwoord |
# |---|---|
# | **Hoeveel RAM kost één afbeelding?** | Baseline: 6272 bytes (6 KB). Eigen encoding: **25 bytes** |
# | **Hoeveel RAM kost 100 afbeeldingen?** | Baseline: ~600 KB (past NIET). Eigen encoding: ~2.5 KB ✓ |
# | **Hoe groot mag het model maximaal zijn?** | 256 KB − input − overhead ≈ **~250 KB** voor model + programma |
# | **Kun je de encoding verder comprimeren?** | Ja: run-length encoding of Huffman op de bits; of nóg kleinere feature-extractie |
# | **Is er informatie verloren gegaan?** | Ja, bij alle verliesgevende methodes (binary, quantization, downscaling) |
# | **Kun je nog steeds het cijfer herkennen?** | Bij de meeste methodes ja – de globale vorm blijft bewaard |
#
# ### Aanbeveling voor het MysteryDevice
# **Eigen encoding (14×14 binary packed = 25 bytes)** is de beste keuze:
# - Minimaal RAM-gebruik
# - Eenvoudige berekening (geen float-bewerkingen)
# - Genoeg structuur voor een Decision Tree om op te classifiseren